# Document extraction with Qwen3-VL

This notebook preprocesses one document and sends it through the provider-neutral `ocr_extraction` client. The local server translates the client contract to Ollama and runs `qwen3-vl:4b-instruct` (the valid Ollama spelling of the requested GWEN3-VL model).

Start the server in another terminal from the repository root:

```bash
$env:PYTHONPATH='ocr/src'
uv run uvicorn ocr_extraction.server:app --host 127.0.0.1 --port 8080
```

In [ ]:
from pathlib import Path
import json
import os
import random
import sys

PACKAGE_DIR = Path('/home/jovyan/ocr')
if not PACKAGE_DIR.exists():
    PACKAGE_DIR = Path('../ocr').resolve()
sys.path.insert(0, str(PACKAGE_DIR / 'src'))

from ocr_extraction import (
    ExtractionRequest,
    HttpExtractorClient,
    HttpExtractorConfig,
    preprocess_document,
)

In [ ]:
SEED = 42
IMAGE_DIR = Path('../data/images')
RUN_DIR = Path('../runs/extractor')
EXPLICIT_IMAGE: Path | None = None

image_paths = sorted(
    p for p in IMAGE_DIR.glob('*')
    if p.suffix.lower() in {'.jpg', '.jpeg', '.png'}
)
if not image_paths:
    raise FileNotFoundError(f'No JPEG or PNG images found in {IMAGE_DIR.resolve()}')
input_path = EXPLICIT_IMAGE or random.Random(SEED).choice(image_paths)
print(f'Selected: {input_path} ({len(image_paths)} images available)')

## Preprocess, then call the extractor package

The notebook owns preprocessing and experiment output. It does not import Ollama or construct provider-specific requests; `HttpExtractorClient` owns the HTTP boundary.

In [ ]:
preprocessed = preprocess_document(input_path)
normalized_path = RUN_DIR / input_path.stem / 'normalized_gray.png'
normalized_path.parent.mkdir(parents=True, exist_ok=True)
import cv2
cv2.imwrite(str(normalized_path), preprocessed.stages['normalized_gray'])
print(json.dumps({k: v for k, v in preprocessed.metadata.items() if k != 'corners'}, indent=2))
print('warnings:', preprocessed.warnings)

In [ ]:
extractor_url = os.getenv('EXTRACTOR_URL', 'http://127.0.0.1:8080')
config = HttpExtractorConfig(
    base_url=extractor_url,
    extraction_path='/v1/extractions',
    timeout_seconds=float(os.getenv('EXTRACTOR_TIMEOUT_SECONDS', '300')),
    max_attempts=2,
)
request = ExtractionRequest(
    document_path=normalized_path,
    prompt='Extract the invoice or document text and fields that are clearly visible.',
    options={'language': 'es', 'model': 'qwen3-vl:4b-instruct'},
)

with HttpExtractorClient(config) as extractor:
    response = extractor.extract(request)

print('request_id:', response.telemetry.request_id)
print('attempts:', response.telemetry.attempts)
print('elapsed_ms:', round(response.telemetry.elapsed_ms, 1))
print('server_processing_ms:', response.telemetry.server_processing_ms)
print(json.dumps(response.result.model_dump(mode='json'), ensure_ascii=False, indent=2))

In [ ]:
artifact_dir = RUN_DIR / input_path.stem
(artifact_dir / 'normalized_extraction.json').write_text(
    json.dumps(response.result.model_dump(mode='json'), ensure_ascii=False, indent=2),
    encoding='utf-8',
)
(artifact_dir / 'raw_response.json').write_text(
    json.dumps(response.raw_response, ensure_ascii=False, indent=2),
    encoding='utf-8',
)
print(f'Artifacts written to {artifact_dir.resolve()}')

## Batch extraction

This is a diagnostic batch. Each document gets its own request ID and normalized/raw artifact pair; failures are recorded without stopping the remaining documents.

In [ ]:
batch = []
with HttpExtractorClient(config) as extractor:
    for path in image_paths:
        try:
            item = preprocess_document(path)
            item_dir = RUN_DIR / path.stem
            item_dir.mkdir(parents=True, exist_ok=True)
            item_path = item_dir / 'normalized_gray.png'
            cv2.imwrite(str(item_path), item.stages['normalized_gray'])
            item_response = extractor.extract(ExtractionRequest(
                document_path=item_path,
                prompt='Extract the visible document text and structured fields.',
                options={'language': 'es'},
            ))
            (item_dir / 'normalized_extraction.json').write_text(
                json.dumps(item_response.result.model_dump(mode='json'), ensure_ascii=False, indent=2),
                encoding='utf-8',
            )
            (item_dir / 'raw_response.json').write_text(
                json.dumps(item_response.raw_response, ensure_ascii=False, indent=2),
                encoding='utf-8',
            )
            batch.append({'image': path.name, 'status': 'ok', 'request_id': item_response.telemetry.request_id, 'elapsed_ms': item_response.telemetry.elapsed_ms})
        except Exception as exc:  # diagnostic batch: retain every failure
            batch.append({'image': path.name, 'status': 'error', 'error': f'{type(exc).__name__}: {exc}'})

print(json.dumps(batch, ensure_ascii=False, indent=2))